# Paso 2: filtrar el audio sintético del TTS

Decide con qué clips de `audio_generado` nos quedamos. Los filtros se aplican **en orden** y se cuenta
cuántos clips descarta cada uno (un clip se cuenta solo en el primer filtro que lo descarta):

| # | filtro | descarta si | por qué |
|---|---|---|---|
| 0 | no generado / fuga | sin wav, error al generar, o su texto contiene una frase de dev/test | sin audio no hay clip; la fuga inflaría el WER |
| 1a | vacío | dura < 0.5 s o RMS < −50 dBFS | el vocoder no produjo habla |
| 1b | saturado | > 0.1 % de muestras en mesetas pegadas al pico | onda recortada (en SK001 y en el TTS sano es 0 %) |
| 1c | silencio | voz < 50 % del clip (Silero VAD) | Tacotron2 que no se detiene genera silencio o ruido largo |
| 1d | QC de atención | el `estado` de la generación no es `ok` | atención difusa, no llegó al final del texto, no se detuvo |
| 2 | duración vs texto | segundos de voz por carácter fuera de p1–p99 de SK001 | corta palabras (muy rápido) o balbucea (muy lento) |

**Decisiones que vale la pena justificar en la tesis**

- En el filtro 2 se miden los **segundos de voz** (VAD), no la duración del archivo, tanto en el TTS como en SK001.
  Los clips reales traen silencio al inicio y al final y los sintéticos casi no, así que comparar duraciones brutas
  marcaría como "rápido" a todo el sintético.
**Pospuesto: ida y vuelta con el ASR.** Transcribir cada clip con el ASR semilla y descartar los de CER alto
se deja para cuando haya un ASR más confiable: el semilla saca WER 52 % / CER 14 % en voces nuevas, y como juez
mezclaría sus propios errores con los del TTS. Los filtros 0–2 no lo necesitan (no usan GPU).

Todo es reanudable: las medidas se guardan en `audio_generado/filtrado/` a medida que se calculan.

In [ ]:
#@title 1. Parámetros
AUDIO_GEN = "/content/drive/MyDrive/TESIS/audio_generado"  #@param {type:"string"}
SPLIT = "/content/drive/MyDrive/TESIS/asr/split_v1.csv"  #@param {type:"string"}
DATASET_ID = ""  #@param {type:"string"}
DRIVE_DATASET = "/content/drive/MyDrive/TESIS/asr/dataset_v1.0.zip"  #@param {type:"string"}

#@markdown **Filtro 1**
DUR_MIN_S = 0.5
RMS_MIN_DB = -50.0
MESETA_MAX_PCT = 0.1
VOZ_MIN_PCT = 50.0  #@param {type:"number"}
USAR_QC_TACOTRON = True  #@param {type:"boolean"}
#@markdown **Filtro 2** (percentiles de SK001)
P_BAJO, P_ALTO = 1, 99
SEED = 1234
SHA_DATASET_PROYECTO = "174b61329123a19a4ed9d09b4d8cb229eb7989cddc9d63d9888859c906993a5d"

## 2. Drive (no hace falta GPU)

In [ ]:
import torch
from google.colab import drive
drive.mount("/content/drive")

## 3. Librerías

In [ ]:
!pip install -q jiwer==4.0.0 silero-vad gdown

## 4. Núcleo (probado localmente)

In [ ]:
%%writefile /content/asr_core.py
# ---- nucleo del ASR: se prueba localmente y se copia tal cual al notebook ----
import hashlib, re, unicodedata
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import soundfile as sf
import torch

SR = 16000
FUENTE_SK = "menendez_gomez_2025"


# ============================ TEXTO ============================
def normalizar(t):
    """Misma normalizacion que build_dataset_v1.py (text_norm): NFC, minusculas, sin puntuacion.
    Se aplica tambien a lo que predice Whisper, que tiende a poner mayusculas y puntos."""
    t = unicodedata.normalize("NFC", str(t)).lower()
    t = re.sub(r"[¡!¿?.,;:…\"“”«»()\[\]{}—–\-_/\\]", " ", t)
    t = t.replace("'", " ").replace("’", " ")
    return re.sub(r"\s+", " ", t).strip()


def sha256(ruta):
    h = hashlib.sha256()
    with open(ruta, "rb") as f:
        for b in iter(lambda: f.read(1 << 20), b""):
            h.update(b)
    return h.hexdigest()


# ============================ SPLIT ============================
def hacer_split(meta, lineas_tts, seed, frac_dev, frac_test, frac_otros):
    """Particion fija para todos los experimentos de la tesis.

    - test_otros: grupos de hablante ENTEROS de youtube y workshop (voces que el modelo no oye).
    - dev / test_sk001: clips de SK001 cuyo texto NO aparece en el txt del TTS, para que los
      experimentos con audio sintetico no vean las frases de evaluacion.
    - Un texto de evaluacion nunca queda tambien en train (se saca de train: 'fuera_texto').
    """
    m = meta.sort_values("id").reset_index(drop=True).copy()
    m["split"] = "train"
    rng = np.random.RandomState(seed)

    # 1) hablantes no vistos
    for fuente in sorted(set(m.source) - {FUENTE_SK}):
        d = m[m.source == fuente].groupby("speaker_group").duration_s.sum()
        grupos = sorted(d.index)
        rng.shuffle(grupos)
        acum, elegidos = 0.0, []
        for g in grupos:
            if acum >= frac_otros * d.sum():
                break
            elegidos.append(g)
            acum += d[g]
        m.loc[m.speaker_group.isin(elegidos), "split"] = "test_otros"

    # 2) SK001: textos que no esten contenidos en ninguna linea del TTS
    blob = "\n".join(f" {normalizar(l)} " for l in lineas_tts if l.strip())
    sk = m[m.source == FUENTE_SK]
    elegibles = sorted(t for t in set(sk.text_norm) if f" {t} " not in blob)
    rng.shuffle(elegibles)
    n_test, n_dev = round(frac_test * len(sk)), round(frac_dev * len(sk))
    cuenta = sk.text_norm.value_counts()
    usados = {"test_sk001": 0, "dev": 0}
    for t in elegibles:
        destino = "test_sk001" if usados["test_sk001"] < n_test else "dev" if usados["dev"] < n_dev else None
        if destino is None:
            break
        m.loc[(m.source == FUENTE_SK) & (m.text_norm == t), "split"] = destino
        usados[destino] += int(cuenta[t])

    # 3) sin fuga por texto: evaluacion con texto en el TTS, o train con texto de evaluacion
    evaluacion = m.split != "train"
    en_tts = m.text_norm.map(lambda t: f" {t} " in blob)
    m.loc[evaluacion & en_tts, "split"] = "fuera_tts"
    textos_eval = set(m.loc[m.split.isin(["dev", "test_sk001", "test_otros"]), "text_norm"])
    m.loc[(m.split == "train") & m.text_norm.isin(textos_eval), "split"] = "fuera_texto"
    return m


def resumen_split(m):
    r = m.groupby("split").agg(clips=("id", "size"), minutos=("duration_s", lambda s: round(s.sum() / 60, 1)),
                               hablantes=("speaker_group", "nunique"))
    return r.reindex([s for s in ["train", "dev", "test_sk001", "test_otros", "fuera_tts", "fuera_texto"]
                      if s in r.index])


# ============================ DATOS PARA WHISPER ============================
class ClipsASR(torch.utils.data.Dataset):
    def __init__(self, df, raiz, processor):
        self.filas = list(df[["file_name", "text_norm"]].itertuples(index=False))
        self.raiz, self.p = Path(raiz), processor

    def __len__(self):
        return len(self.filas)

    def __getitem__(self, i):
        archivo, texto = self.filas[i]
        y, sr = sf.read(self.raiz / archivo, dtype="float32")
        assert sr == SR, f"{archivo}: {sr} Hz"
        return {"input_features": self.p.feature_extractor(y, sampling_rate=SR).input_features[0],
                "labels": self.p.tokenizer(texto).input_ids}


@dataclass
class Collator:
    processor: object
    decoder_start_token_id: int

    def __call__(self, feats):
        batch = self.processor.feature_extractor.pad(
            [{"input_features": f["input_features"]} for f in feats], return_tensors="pt")
        lab = self.processor.tokenizer.pad([{"input_ids": f["labels"]} for f in feats], return_tensors="pt")
        labels = lab["input_ids"].masked_fill(lab["attention_mask"].ne(1), -100)
        if (labels[:, 0] == self.decoder_start_token_id).all():   # el modelo lo agrega solo
            labels = labels[:, 1:]
        batch["labels"] = labels
        return batch


# ============================ METRICAS ============================
def wer_cer(refs, hyps):
    import jiwer
    pares = [(r, h) for r, h in zip(refs, hyps) if r]            # jiwer no acepta referencias vacias
    r, h = [p[0] for p in pares], [p[1] for p in pares]
    return 100 * jiwer.wer(r, h), 100 * jiwer.cer(r, h)


def metricas_trainer(processor):
    tok = processor.tokenizer

    def f(pred):
        ids, lab = pred.predictions, pred.label_ids
        lab = np.where(lab == -100, tok.pad_token_id, lab)
        ids = np.where(ids == -100, tok.pad_token_id, ids)
        hyps = [normalizar(t) for t in tok.batch_decode(ids, skip_special_tokens=True)]
        refs = [normalizar(t) for t in tok.batch_decode(lab, skip_special_tokens=True)]
        w, c = wer_cer(refs, hyps)
        return {"wer": round(w, 2), "cer": round(c, 2)}
    return f


def wer_con_ic(pred, seed=0, n=1000):
    """WER corpus (no promedio por clip) e IC 95% por bootstrap sobre clips."""
    import jiwer
    p = pred[pred.text_norm.str.len() > 0]
    err, pal = [], []
    for a, b in zip(p.text_norm, p.prediccion):
        o = jiwer.process_words(a, b)
        err.append(o.substitutions + o.deletions + o.insertions)
        pal.append(o.substitutions + o.deletions + o.hits)
    err, pal = np.array(err), np.array(pal)
    rng = np.random.RandomState(seed)
    k = rng.randint(0, len(p), size=(n, len(p)))
    boot = 100 * err[k].sum(1) / pal[k].sum(1)
    return dict(wer=round(float(100 * err.sum() / pal.sum()), 2), ic95=[round(float(x), 2) for x in np.percentile(boot, [2.5, 97.5])],
                cer=round(100 * jiwer.cer(list(p.text_norm), list(p.prediccion)), 2), clips=int(len(p)),
                palabras=int(pal.sum()))


@torch.no_grad()
def transcribir(model, processor, df, raiz, device, batch=16, max_tokens=128, num_beams=1):
    """Transcribe df y devuelve una copia con la prediccion y el error por clip."""
    import jiwer
    model.eval()
    dtype = next(model.parameters()).dtype
    salida = []
    for i in range(0, len(df), batch):
        trozo = df.iloc[i:i + batch]
        audios = [sf.read(Path(raiz) / f, dtype="float32")[0] for f in trozo.file_name]
        x = processor.feature_extractor(audios, sampling_rate=SR, return_tensors="pt").input_features
        ids = model.generate(x.to(device, dtype), language="spanish", task="transcribe",
                             max_new_tokens=max_tokens, num_beams=num_beams)
        salida += processor.tokenizer.batch_decode(ids, skip_special_tokens=True)
    r = df[["id", "file_name", "source", "speaker_group", "duration_s", "text_norm"]].copy()
    r["prediccion"] = [normalizar(t) for t in salida]
    r["wer_clip"] = [round(100 * jiwer.wer(a, b), 1) if a else None for a, b in zip(r.text_norm, r.prediccion)]
    return r

In [ ]:
%%writefile /content/tts_filtro.py
# ---- filtros del audio sintetico: se prueba localmente y se copia tal cual al notebook ----
import csv, math, os, unicodedata
from math import gcd
from pathlib import Path

import numpy as np
import soundfile as sf
import torch
from scipy.signal import resample_poly

SR = 16000


def a16k(y, sr):
    if y.ndim > 1:
        y = y.mean(axis=1)
    if sr != SR:
        g = gcd(sr, SR)
        y = resample_poly(y, SR // g, sr // g)
    return np.ascontiguousarray(y, dtype=np.float32)


def muestras_en_meseta(y, run=3, tol=0.999):
    """Muestras en tramos de >= run muestras seguidas pegadas al pico: la firma de una onda recortada.
    Un pico natural toca el maximo en 1-2 muestras; una saturada se queda plana ahi."""
    a = np.abs(y)
    pico = a.max() if a.size else 0.0
    if pico <= 0:
        return 0
    m = np.concatenate([[0], (a >= tol * pico).astype(np.int8), [0]])
    d = np.diff(m)
    ini, fin = np.where(d == 1)[0], np.where(d == -1)[0]
    largos = fin - ini
    return int(largos[largos >= run].sum())


def medir(ruta, vad, get_ts):
    """Medidas de audio de un clip. voz_s = segundos con voz segun Silero VAD."""
    try:
        y, sr = sf.read(ruta, dtype="float32", always_2d=True)
    except Exception as e:
        return dict(error=f"{type(e).__name__}: {e}"[:120])
    y = y.mean(axis=1)
    dur = len(y) / sr
    if len(y) == 0:
        return dict(dur_s=0.0, rms_db=-120.0, meseta_pct=0.0, voz_s=0.0, voz_pct=0.0)
    rms = float(np.sqrt(np.mean(y.astype(np.float64) ** 2)))
    y16 = a16k(y, sr)
    ts = get_ts(torch.from_numpy(y16), vad, sampling_rate=SR) if len(y16) >= 512 else []
    voz = sum(t["end"] - t["start"] for t in ts) / SR
    return dict(dur_s=round(dur, 3), rms_db=round(20 * math.log10(rms + 1e-12), 2),
                meseta_pct=round(100 * muestras_en_meseta(y) / len(y), 4),
                voz_s=round(voz, 3), voz_pct=round(100 * voz / dur, 1) if dur else 0.0)


def medir_lote(filas, raiz, cache, vad, get_ts, cada=200):
    """filas: [(id, ruta_relativa)]. Reanudable: lo ya medido en `cache` no se repite."""
    hechos = set()
    if Path(cache).exists():
        with open(cache, encoding="utf-8", newline="") as f:
            hechos = {r["id"] for r in csv.DictReader(f) if r.get("id")}
    pend = [f for f in filas if f[0] not in hechos]
    campos = ["id", "dur_s", "rms_db", "meseta_pct", "voz_s", "voz_pct", "error"]
    nuevo = not Path(cache).exists()
    from tqdm.auto import tqdm
    with open(cache, "a", encoding="utf-8", newline="") as f:
        w = csv.DictWriter(f, fieldnames=campos)
        if nuevo:
            w.writeheader()
        for k, (i, rel) in enumerate(tqdm(pend, desc="midiendo")):
            w.writerow({"id": i, **medir(os.path.join(raiz, rel), vad, get_ts)})
            if k % cada == 0:
                f.flush()
    return len(pend)


# ============================ TEXTO / CER ============================
_ACENTOS = {"́", "̀"}


def sin_tildes(t):
    t = unicodedata.normalize("NFD", str(t))
    return unicodedata.normalize("NFC", "".join(c for c in t if c not in _ACENTOS))


def cer_clip(ref, hyp):
    import jiwer
    if not ref:
        return None
    return round(100 * jiwer.cer(ref, hyp if hyp else ""), 2)


def leer_16k(ruta):
    y, sr = sf.read(ruta, dtype="float32", always_2d=True)
    return a16k(y, sr)


@torch.no_grad()
def transcribir_audios(model, processor, audios, device, max_tokens=128):
    dtype = next(model.parameters()).dtype
    x = processor.feature_extractor(audios, sampling_rate=SR, return_tensors="pt").input_features
    ids = model.generate(x.to(device, dtype), language="spanish", task="transcribe",
                         max_new_tokens=max_tokens, num_beams=1)
    return processor.tokenizer.batch_decode(ids, skip_special_tokens=True)


def orden_de_trabajo(df, seed, bloque=2000):
    """Barajado con semilla (lo transcrito en cualquier momento es una muestra representativa) y, dentro de
    cada bloque, ordenado por largo del texto: un lote dura lo que su frase más larga, así se decodifica menos relleno."""
    d = df.sample(frac=1, random_state=seed).reset_index(drop=True)
    d["_b"] = d.index // bloque
    d["_n"] = d.text_norm.str.len()
    return d.sort_values(["_b", "_n"]).drop(columns=["_b", "_n"])


def ida_y_vuelta(df, raiz, cache, model, processor, device, normalizar, batch=32, cada=10, seed=1234, hilos=16):
    """df con id, archivo, text_norm. Escribe id, prediccion, cer, cer_sin_tildes. Reanudable.
    Lee los wavs en paralelo y prepara el lote siguiente mientras la GPU decodifica el actual:
    leer de Drive archivo por archivo deja la GPU esperando."""
    from concurrent.futures import ThreadPoolExecutor
    import time
    hechos = set()
    if Path(cache).exists():
        with open(cache, encoding="utf-8", newline="") as f:
            hechos = {r["id"] for r in csv.DictReader(f) if r.get("id")}
    pend = orden_de_trabajo(df[~df.id.isin(hechos)], seed)
    lotes = [pend.iloc[i:i + batch] for i in range(0, len(pend), batch)]
    campos = ["id", "prediccion", "cer", "cer_sin_tildes"]
    nuevo = not Path(cache).exists()
    from tqdm.auto import tqdm
    model.eval()
    io_pool, pre = ThreadPoolExecutor(hilos), ThreadPoolExecutor(1)
    cargar = lambda t: list(io_pool.map(leer_16k, [os.path.join(raiz, a) for a in t.archivo]))
    with open(cache, "a", encoding="utf-8", newline="") as f:
        w = csv.DictWriter(f, fieldnames=campos)
        if nuevo:
            w.writeheader()
        barra = tqdm(lotes, desc="ida y vuelta")
        fut = pre.submit(cargar, lotes[0]) if lotes else None
        for k, t in enumerate(barra):
            t0 = time.time()
            audios = fut.result()
            espera = time.time() - t0
            if k + 1 < len(lotes):
                fut = pre.submit(cargar, lotes[k + 1])
            t1 = time.time()
            hyps = transcribir_audios(model, processor, audios, device)
            barra.set_postfix(espera_lectura=f"{espera:.1f}s", gpu=f"{time.time() - t1:.1f}s")
            for r, h in zip(t.itertuples(), hyps):
                h = normalizar(h)
                w.writerow(dict(id=r.id, prediccion=h, cer=cer_clip(r.text_norm, h),
                                cer_sin_tildes=cer_clip(sin_tildes(r.text_norm), sin_tildes(h))))
            if k % cada == 0:
                f.flush()
    io_pool.shutdown(); pre.shutdown()
    return len(pend)

## 5. Datos: manifest de la generación, dataset real y partición

In [ ]:
import sys, importlib, json, re, shutil, zipfile
sys.path.insert(0, "/content")
import asr_core as C, tts_filtro as T
importlib.reload(C); importlib.reload(T)
import numpy as np, pandas as pd
from pathlib import Path

GEN = Path(AUDIO_GEN)
FIL = GEN / "filtrado"
FIL.mkdir(exist_ok=True)

# --- manifest de la generación (misma limpieza que usa generation.ipynb al reanudar)
man = pd.read_csv(GEN / "manifest.csv", dtype=str, keep_default_na=False)
man = man[man.id.str.fullmatch(r"L[0-9]{5}_[0-9]{2}")].drop_duplicates("id", keep="last").reset_index(drop=True)
man["text_norm"] = man.texto.map(C.normalizar)
man["n_chars"] = man.text_norm.str.replace(" ", "").str.len()
print(f"manifest: {len(man)} clips generados | estados: {man.estado.str.split(',').str[0].value_counts().to_dict()}")

# --- dataset real (para SK001 y para los textos de evaluación)
def id_de(v):
    v = v.strip()
    m = re.search(r"/d/([-\w]{20,})", v) or re.search(r"[?&]id=([-\w]{20,})", v)
    return m.group(1) if m else v

if DATASET_ID.strip():
    import gdown
    ZIP = Path(gdown.download(id=id_de(DATASET_ID), output="/content/", quiet=True) or "")
else:
    ZIP = Path(DRIVE_DATASET)
if not ZIP.is_file():
    raise SystemExit(f"No encuentro el dataset ({ZIP}). Pon DATASET_ID o corrige DRIVE_DATASET.")
sha_zip = C.sha256(ZIP)
print("dataset:", "= el del proyecto" if sha_zip == SHA_DATASET_PROYECTO else f"DISTINTO ({sha_zip[:12]})")
DATOS = Path("/content/datos") / sha_zip[:12]
if not DATOS.exists():
    tmp = DATOS.with_name(DATOS.name + ".tmp"); shutil.rmtree(tmp, ignore_errors=True)
    with zipfile.ZipFile(ZIP) as z:
        z.extractall(tmp)
    tmp.rename(DATOS)
RAIZ = next(DATOS.rglob("metadata.csv")).parent
meta = pd.read_csv(RAIZ / "metadata.csv")
split = meta.merge(pd.read_csv(SPLIT)[["id", "split"]], on="id")

# --- fuga: textos de dev/test que aparezcan dentro de un clip sintético
textos_eval = set(split[split.split.isin(["dev", "test_sk001", "test_otros"])].text_norm)
textos_train = set(split[split.split == "train"].text_norm)
man["fuga_eval"] = man.text_norm.map(lambda t: any(f" {e} " in f" {t} " for e in textos_eval))
man["texto_en_train_real"] = man.text_norm.isin(textos_train)
print(f"clips con texto de evaluación (fuga): {int(man.fuga_eval.sum())} | "
      f"con texto idéntico a una frase real de train: {int(man.texto_en_train_real.sum())}")

## 6. Medidas de audio de cada clip sintético (filtro 1)

Silero VAD en CPU, ~70× tiempo real. Reanudable: si se corta, vuelve a ejecutar.

In [ ]:
from silero_vad import load_silero_vad, get_speech_timestamps
torch.set_num_threads(2)
vad = load_silero_vad()

con_wav = man[(man.archivo != "") & ~man.estado.str.startswith("error")]
n = T.medir_lote(list(zip(con_wav.id, con_wav.archivo)), str(GEN), FIL / "medidas_tts.csv", vad, get_speech_timestamps)
print(f"medidos ahora: {n}")
med = pd.read_csv(FIL / "medidas_tts.csv", keep_default_na=False).drop_duplicates("id", keep="last")
d = man.drop(columns=[c for c in med.columns if c != "id" and c in man]).merge(med, on="id", how="left")
for c in ("dur_s", "rms_db", "meseta_pct", "voz_s", "voz_pct"):
    d[c] = pd.to_numeric(d[c], errors="coerce")
d["seg_por_char"] = d.voz_s / d.n_chars.replace(0, np.nan)
print(d[["dur_s", "rms_db", "meseta_pct", "voz_pct", "seg_por_char"]].describe(percentiles=[.01, .5, .99]).round(3).to_string())

## 7. Referencia SK001: segundos de voz por carácter (filtro 2)

In [ ]:
sk = meta[meta.source == "menendez_gomez_2025"].copy()
T.medir_lote(list(zip(sk.id, sk.file_name)), str(RAIZ), FIL / "medidas_sk001.csv", vad, get_speech_timestamps)
msk = pd.read_csv(FIL / "medidas_sk001.csv").drop_duplicates("id", keep="last")
sk = sk.merge(msk, on="id")
sk["seg_por_char"] = sk.voz_s / sk.text_norm.str.replace(" ", "").str.len()
LIM_BAJO, LIM_ALTO = np.percentile(sk.seg_por_char.dropna(), [P_BAJO, P_ALTO])
print(f"SK001 ({len(sk)} clips): seg/carácter p{P_BAJO} = {LIM_BAJO:.4f}  mediana = {sk.seg_por_char.median():.4f}  "
      f"p{P_ALTO} = {LIM_ALTO:.4f}")
print(f"TTS: mediana = {d.seg_por_char.median():.4f}")

import matplotlib.pyplot as plt
plt.figure(figsize=(8, 3))
plt.hist(sk.seg_por_char, bins=60, alpha=.6, density=True, label="SK001 real")
plt.hist(d.seg_por_char.dropna().clip(upper=0.3), bins=60, alpha=.6, density=True, label="TTS")
for x in (LIM_BAJO, LIM_ALTO):
    plt.axvline(x, color="k", ls="--")
plt.xlabel("segundos de voz por carácter"); plt.legend(); plt.tight_layout()
plt.savefig(FIL / "seg_por_caracter.png", dpi=120); plt.show()

## 8. Filtros 0–2

In [ ]:
d["descarte"] = ""

def aplicar(nombre, mascara):
    m = (d.descarte == "") & mascara.fillna(False).astype(bool)
    d.loc[m, "descarte"] = nombre
    return int(m.sum())

ETAPAS = [
    ("0 no generado", (d.archivo == "") | d.estado.str.startswith("error") | d.dur_s.isna()
                      | (d["error"].fillna("").astype(str) != "")),
    ("0 fuga de evaluación", d.fuga_eval),
    ("1a vacío", (d.dur_s < DUR_MIN_S) | (d.rms_db < RMS_MIN_DB)),
    ("1b saturado", d.meseta_pct > MESETA_MAX_PCT),
    ("1c silencio > 50%", d.voz_pct < VOZ_MIN_PCT),
    ("1d QC atención Tacotron2", (d.estado != "ok") if USAR_QC_TACOTRON else pd.Series(False, index=d.index)),
    ("2 duración vs texto", (d.seg_por_char < LIM_BAJO) | (d.seg_por_char > LIM_ALTO)),
]
for nombre, m in ETAPAS:
    print(f"{nombre:28s} descarta {aplicar(nombre, m):6d}")
print(f"aceptados: {(d.descarte == '').sum()} de {len(d)}")

## 9. Tabla resumen y `metadata_tts.csv`

Si cambias un umbral en la celda 1, ejecuta la celda 8 y luego esta (las medidas ya están guardadas).

In [ ]:
# --- tabla resumen
h = lambda m: round(d.loc[m, "dur_s"].sum() / 3600, 2)
filas = [dict(etapa="generados", descartados="", quedan=len(d), horas_que_quedan=h(d.descarte.notna()))]
fuera = set()
for nombre in [e[0] for e in ETAPAS]:
    fuera.add(nombre)
    sigue = ~d.descarte.isin(fuera)
    filas.append(dict(etapa=nombre, descartados=int((d.descarte == nombre).sum()), quedan=int(sigue.sum()),
                      horas_que_quedan=h(sigue)))
acept = d[d.descarte == ""]
filas.append(dict(etapa="ACEPTADOS", descartados=len(d) - len(acept), quedan=len(acept), horas_que_quedan=h(d.descarte == "")))
resumen = pd.DataFrame(filas)
resumen.to_csv(FIL / "resumen_filtrado.csv", index=False)
print(resumen.to_string(index=False))

# --- salidas
d.to_csv(FIL / "detalle_filtrado.csv", index=False)
out = pd.DataFrame({
    "id": "TTS-" + acept.id, "file_name": acept.archivo, "transcription": acept.texto, "text_norm": acept.text_norm,
    "source": "tts_sintetico", "subsource": "tacotron2_hifigan_SKF2", "speaker_group": "TTS-SK001",
    "duration_s": acept.dur_s, "voz_pct": acept.voz_pct, "seg_por_char": acept.seg_por_char.round(4),
    "seed_tts": acept.seed, "texto_en_train_real": acept.texto_en_train_real,
    "linea_origen": acept.id.str[1:6].astype(int)})
out.to_csv(GEN / "metadata_tts.csv", index=False)
cfg = {k: v for k, v in globals().items() if k.isupper() and isinstance(v, (int, float, str, bool))}
cfg.update(lim_seg_por_char=[float(LIM_BAJO), float(LIM_ALTO)], sha256_dataset=sha_zip,
           generados=len(d), aceptados=len(acept), horas_aceptadas=h(d.descarte == ""))
(FIL / "config_filtrado.json").write_text(json.dumps(cfg, indent=2, ensure_ascii=False))
print(f"\n{GEN / 'metadata_tts.csv'}: {len(out)} clips, {out.duration_s.sum() / 3600:.2f} h")